#### Cuaderno que permite revisar el pre-procesamiento

In [14]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_split import load_raw_data, split_data

df = load_raw_data()
X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (120000, 10) | X_test: (30000, 10)


In [15]:
print("NaN en X_train (antes):")
print(X_train.isna().sum())
print("\nNaN totales:", X_train.isna().sum().sum())

NaN en X_train (antes):
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           23675
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3128
dtype: int64

NaN totales: 26803


In [16]:
from src.features.preprocessing import build_preprocessor

prep = build_preprocessor()
prep.fit(X_train)                     #aprende las medianas solo del train

X_train_prep = prep.transform(X_train)
X_test_prep = prep.transform(X_test)

# Volver a DataFrame solo para inspeccionar cómodamente
X_train_prep = pd.DataFrame(X_train_prep, columns=X_train.columns, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=X_test.columns, index=X_test.index)

print("X_train procesado:", X_train_prep.shape)
print("X_test procesado :", X_test_prep.shape)

X_train procesado: (120000, 10)
X_test procesado : (30000, 10)


In [17]:
#verificación que no quedne faltantes ni anomalías
print("NaN restantes en train:", X_train_prep.isna().sum().sum())
print("NaN restantes en test :", X_test_prep.isna().sum().sum())
print("Edad mínima en train  :", X_train_prep["age"].min())

late_cols = ["NumberOfTime30-59DaysPastDueNotWorse",
             "NumberOfTimes90DaysLate",
             "NumberOfTime60-89DaysPastDueNotWorse"]
print("Máximos en variables de atraso (ya sin 96/98):")
print(X_train_prep[late_cols].max())

NaN restantes en train: 0
NaN restantes en test : 0
Edad mínima en train  : 21.0
Máximos en variables de atraso (ya sin 96/98):
NumberOfTime30-59DaysPastDueNotWorse    13.0
NumberOfTimes90DaysLate                 17.0
NumberOfTime60-89DaysPastDueNotWorse    11.0
dtype: float64


In [18]:
#resumen estadístico del x train procesado
print(X_train_prep.describe().T[["min", "50%", "max"]])

                                       min          50%        max
RevolvingUtilizationOfUnsecuredLines   0.0     0.153318    50708.0
age                                   21.0    52.000000      109.0
NumberOfTime30-59DaysPastDueNotWorse   0.0     0.000000       13.0
DebtRatio                              0.0     0.366194   329664.0
MonthlyIncome                          0.0  5390.000000  3008750.0
NumberOfOpenCreditLinesAndLoans        0.0     8.000000       58.0
NumberOfTimes90DaysLate                0.0     0.000000       17.0
NumberRealEstateLoansOrLines           0.0     1.000000       54.0
NumberOfTime60-89DaysPastDueNotWorse   0.0     0.000000       11.0
NumberOfDependents                     0.0     0.000000       20.0
